# Laboratorio 6: Visualización & DataViz

Construir un conjunto de datos mediante web scraping pierde valor si no podemos analizarlo y comunicarlo de forma efectiva. En este laboratorio, nos enfocaremos en estructurar y presentar visualmente la base de reseñas literarias de Bill Gates, extraída en el Laboratorio 4.

A través de **Matplotlib** y **Seaborn**, implementaremos las mejores prácticas del diseño de información para la ciencia de datos. Esto implica crear visualizaciones de "calidad de publicación" mediante la interfaz orientada a objetos (OO), asegurando claridad, legibilidad y accesibilidad (por ejemplo, diseñando para el daltonismo).

## 1. Principios de Diseño y Preparación del Entorno

Previo a dibujar nuestro primer gráfico, estableceremos las bases de un entorno profesional de visualización:
- **Interfaz Orientada a Objetos** (`fig, ax = plt.subplots()`): Otorga control absoluto sobre cada elemento del gráfico (ejes, títulos, cuadrícula), a diferencia de la notación básica `plt.plot()`.
- **Configuración central de Seaborn**: Emplearemos `sns.set_theme()` para definir globalmente el estilo visual, reduciendo el código repetitivo.
- **Data-Ink Ratio**: Minimizaremos los elementos visuales que no aporten datos (como bordes innecesarios, eliminados con `sns.despine()`).
- **Accesibilidad**: Emplearemos paletas diseñadas científicamente para ser legibles por personas con alteraciones en la percepción del color (colorblindness).

In [13]:

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from collections import Counter
import re
import json

# Configuramos el estilo globalmente utilizando Seaborn
sns.set_theme(
    style="ticks",          # Estilo minimalista con marcas en los ejes
    context="notebook",     # Escala de elementos optimizada para notebooks
    palette="colorblind",   # Paleta de colores accesible
    font_scale=1.1          # Aumento ligero de la tipografía para mejor lectura
)

# Extraemos la paleta "colorblind" como lista para asignaciones manuales de color
PALETA = sns.color_palette("colorblind")

print("Se ha configurado exitosamente el entorno gráfico profesional.")

Se ha configurado exitosamente el entorno gráfico profesional.


## 2. Carga y Exploración del Corpus de Datos

Iniciaremos cargando nuestro conjunto de datos, `gatesnotes_books.json`, generado dinámicamente con Scrapling y Playwright. Utilizaremos la biblioteca **Pandas** para convertirlo en un *DataFrame*, una estructura tabular optimizada para el análisis de datos.

In [14]:
import os

# Verificamos la existencia del archivo de datos
ruta_corpus = "gatesnotes_books.json"

if not os.path.exists(ruta_corpus):
    print(f"Error: No se encontró el archivo '{ruta_corpus}'. Asegúrese de haber ejecutado el script en el Laboratorio 4.")
else:
    with open(ruta_corpus, 'r', encoding='utf-8') as f:
        datos = json.load(f)
        
    df = pd.DataFrame(datos)
    print(f"El corpus contiene {len(df)} reseñas estructuradas.")
    display(df.head(3))

Error: No se encontró el archivo 'gatesnotes_books.json'. Asegúrese de haber ejecutado el script en el Laboratorio 4.


## 3. Visualización 1: Gráfico de Barras Horizontales (Atributos Categóricos)

El primer objetivo descriptivo es analizar la distribución temática de las reseñas. Para comparar múltiples categorías con nombres de longitud considerable, un **gráfico de barras horizontales** es la opción superior, garantizando la legibilidad sin necesidad de rotar el texto del eje.

> [!TIP]
> En la visualización analítica, siempre ordene las barras de mayor a menor frecuencia (rankings) a menos que la categoría disponga de un orden intrínseco (como meses o años). A su vez, asigne un "color de acento" al valor principal para guiar el ojo del lector.

In [15]:
if 'df' in locals():
    # 1. Preparación de los datos: Frecuencias de categorías, limitadas al top 10
    categorias = df[df['category'] != '']['category'].value_counts().head(10).reset_index()
    categorias.columns = ['Categoría', 'Cantidad']
    
    # 2. Creación de la figura y los ejes (Interfaz Orientada a Objetos)
    fig, ax = plt.subplots(figsize=(10, 6), constrained_layout=True)
    
    # 3. Diseño del color: El líder se destaca, el resto adopta un tono neutro
    colores = ['#b0b0b0'] * len(categorias)
    if len(colores) > 0:
        colores[0] = PALETA[0]  # Azul accesible de la paleta 'colorblind'
    
    # 4. Generación del gráfico con Seaborn
    sns.barplot(
        data=categorias,
        y='Categoría',
        x='Cantidad',
        hue='Categoría',         # Requerido en las versiones modernas de sns para asignar la paleta
        palette=colores,
        orient='h',
        order=categorias['Categoría'], # Aseguramos el orden descendente
        ax=ax,
        legend=False
    )
    
    # 5. Anotación directa: Mostramos el valor numérico en el extremo de la barra
    for container in ax.containers:
        ax.bar_label(container, fmt='%.0f', padding=5, fontweight='bold', fontsize=11, color='#333333')
        
    # 6. Limpieza y refinamiento visual (Data-Ink ratio)
    ax.set_xlabel('')
    ax.set_ylabel('')
    ax.set_xticks([]) # Ocultamos el eje temporal X, ya anotamos las barras
    sns.despine(left=True, bottom=True) # Retiramos bordes
    
    # 7. Título informativo
    ax.set_title('Distribución temática en las lecturas recomendadas por Bill Gates',
                 fontsize=15, fontweight='bold', pad=15, loc='left')
    
    plt.show()

## 4. Visualización 2: Lollipop Chart (Gráfico de Piruleta)

Para analizar los autores más frecuentes (excluyendo al propio Bill Gates), la diferencia entre valores absolutos suele ser reducida (ej. 2 a 4 libros). Utilizar barras sólidas y gruesas genera un peso visual excesivo y redundante.

Una alternativa sofisticada es el **Lollipop Chart** (Gráfico de Piruleta). Consta de una línea delgada rematada por un punto (*marker*). Esto minimiza el uso de "tinta" (data-ink ratio) aumentando la elegancia sin sacrificar claridad.

In [16]:
if 'df' in locals():
    # 1. Preparación de los datos: Frecuencias de autores
    autores = df[(df['author'] != '') & (df['author'] != 'Bill Gates')]['author'].value_counts()
    autores_recurrentes = autores[autores >= 2].sort_values(ascending=True) # Ascendente para orden de dibujo (abajo hacia arriba)
    
    # 2. Creación del lienzo
    fig, ax = plt.subplots(figsize=(10, 5), constrained_layout=True)
    
    # 3. Dibujo de la estructura del Lollipop
    # La "línea" (cuerpo)
    ax.hlines(
        y=autores_recurrentes.index,
        xmin=0,
        xmax=autores_recurrentes.values,
        color='#b0b0b0',
        linewidth=2.5,
        zorder=1 # Garantizamos que la línea quede por detrás del punto
    )
    
    # El "punto" (cabeza)
    ax.plot(
        autores_recurrentes.values,
        autores_recurrentes.index,
        'o',
        markersize=10,
        color=PALETA[2],  # Verde accesible
        zorder=2
    )
    
    # 4. Anotación directa
    for i, (val, nombre) in enumerate(zip(autores_recurrentes.values, autores_recurrentes.index)):
        ax.text(val + 0.15, i, str(val), va='center', fontweight='bold', fontsize=11)
        
    # 5. Limpieza visual
    ax.set_xlabel('')
    ax.set_ylabel('')
    ax.set_xticks([])
    sns.despine(left=True, bottom=True)
    
    ax.set_title('Autores recurrentes en las reseñas (Múltiples apariciones)',
                 fontsize=15, fontweight='bold', pad=15, loc='left')
    
    plt.show()

## 5. Visualización 3: Barplot de Precisión vs. WordClouds

Cuando buscamos entender la frecuencia de uso de palabras en una serie de textos (por ejemplo, los títulos), es tentador recurrir a un *WordCloud* (Nube de palabras). Sin embargo, en un entorno profesional **orientado a la toma de decisiones precisas**, el área del texto y los ángulos arbitrarios de la nube de palabras dificultan drásticamente la comparación exacta. 

Por ello, extraeremos las palabras, limpiaremos las irrelevantes (*stop words*) y utilizaremos un Barplot estándar.

> [!IMPORTANT]  
> Utilizaremos expresiones regulares (`re`) y el objeto `Counter` nativo de Python para el conteo de frecuencias en reemplazo de analizadores estadísticos complejos, manteniendo el control directo sobre los resultados.

In [17]:
if 'df' in locals():
    # Lista de stop-words básicas en inglés (palabras vacías)
    stopwords = {'the', 'and', 'to', 'of', 'a', 'in', 'is', 'for', 'that', 'on',
                 'we', 'with', 'it', 'how', 'you', 'are', 'what', 'as', 'my', 'i'}

    # 1. Extracción y conteo de palabras en títulos
    todas_palabras = []
    for titulo in df['title']:
        # Pasamos a minúsculas y extraemos solo caracteres alfanuméricos
        palabras = re.findall(r'\b[a-z]{3,}\b', str(titulo).lower())
        todas_palabras.extend([p for p in palabras if p not in stopwords])

    frecuencias = Counter(todas_palabras)
    top_palabras = pd.DataFrame(frecuencias.most_common(15), columns=['Palabra', 'Frecuencia'])

    # 2. Generación del gráfico
    fig, ax = plt.subplots(figsize=(10, 6), constrained_layout=True)
    
    sns.barplot(
        data=top_palabras,
        y='Palabra',
        x='Frecuencia',
        hue='Palabra',
        palette='viridis', # Aplicamos una paleta continua para destacar variación leve
        ax=ax,
        legend=False
    )
    
    # Anotamos los valores precisos al final de cada barra
    for container in ax.containers:
        ax.bar_label(container, fmt='%.0f', padding=4, color='#333333', fontsize=10, fontweight='bold')
        
    ax.set_xlabel('')
    ax.set_ylabel('')
    ax.set_xticks([])
    sns.despine(left=True, bottom=True)
    
    ax.set_title('Palabras de mayor frecuencia identificadas en los títulos',
                 fontsize=15, fontweight='bold', pad=15, loc='left')
    
    plt.show()

## Desafío Práctico

En lugar de utilizar una paleta arcoíris continua para toda la distribución de palabras, intente generar una optimización en el último gráfico orientada al **Data-Ink Ratio** de jerarquías. 

1. Identifique el bloque de código responsable de focalizar colores que empleamos en la Visualización 1.
2. Aplique la técnica para que únicamente la **palabra de frecuencia máxima** resalte utilizando un Color de Acento (ejemplo: `PALETA[3]` que es rojo), dejando el resto gris neutral (`#b0b0b0`).
3. Renderice la versión perfeccionada en la celda siguiente.

> [!TIP]
> Para aplicar correctamente la validación condicional de color en Seaborn, defina y empaquete un vector de múltiples iterables de colores (una lista `colores_ejemplo = [...]`) y asígnela directamente al parámetro `palette=colores_ejemplo` al graficar.

In [2]:
# EJERCICIO: Escriba aquí el bloque de código para implementar el Desafío Práctico.

if 'df' in locals():
    # 1. Extracción de palabras (mismo proceso que Visualización 3)
    stopwords = {'the', 'and', 'to', 'of', 'a', 'in', 'is', 'for', 'that', 'on',
                 'we', 'with', 'it', 'how', 'you', 'are', 'what', 'as', 'my', 'i'}
    
    todas_palabras = []
    for titulo in df['title']:
        palabras = re.findall(r'\b[a-z]{3,}\b', str(titulo).lower())
        todas_palabras.extend([p for p in palabras if p not in stopwords])
    
    frecuencias = Counter(todas_palabras)
    top_palabras = pd.DataFrame(frecuencias.most_common(15), columns=['Palabra', 'Frecuencia'])
    
    # 2. Optimización Data-Ink Ratio: color de acento solo para el máximo
    max_frecuencia = top_palabras['Frecuencia'].max()
    colores_jerarquia = []
    for freq in top_palabras['Frecuencia']:
        if freq == max_frecuencia:
            colores_jerarquia.append(PALETA[3])  # Rojo accesible para el líder
        else:
            colores_jerarquia.append('#b0b0b0')  # Gris neutral para el resto
    
    # 3. Generación del gráfico optimizado
    fig, ax = plt.subplots(figsize=(10, 6), constrained_layout=True)
    
    sns.barplot(
        data=top_palabras,
        y='Palabra',
        x='Frecuencia',
        hue='Palabra',
        palette=colores_jerarquia,
        ax=ax,
        legend=False
    )
    
    # 4. Anotación directa
    for container in ax.containers:
        ax.bar_label(container, fmt='%.0f', padding=4, color='#333333', fontsize=10, fontweight='bold')
    
    # 5. Limpieza visual (Data-Ink ratio)
    ax.set_xlabel('')
    ax.set_ylabel('')
    ax.set_xticks([])
    sns.despine(left=True, bottom=True)
    
    ax.set_title('Palabras de mayor frecuencia (con focalización de acento)',
                 fontsize=15, fontweight='bold', pad=15, loc='left')
    
    plt.show()
    print("Desafío práctico completado: La palabra más frecuente ahora destaca con color de acento.")